In [5]:
!pip install langchain-text-splitters
!pip install langchain-openai
!pip install langchain_classic
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260319_RAG_Document.ipynb)

In [ ]:
import os

import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from sklearn.metrics.pairwise import cosine_similarity

from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small', api_key=api_key)

In [ ]:
# path 라이브러리, 현재 경로를 쉽게 가져옴
SAMPLE_DIR = Path('sample_data') # 패스 생성
SAMPLE_DIR.mkdir(exist_ok=True) #디렉토리 생성

In [ ]:
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}

In [ ]:
for filename, content in sample_texts.items():
  (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
  # E.g. sample_data/product_manual.txt

In [ ]:
csv_data = [
    {'이름': '김철수', '부서': '개발1팀', '직급': '대리'},
    {'이름': '김민아', '부서': '개발1팀', '직급': '대리'},
    {'이름': '박지민', '부서': '개발1팀', '직급': '대리'},
]

with open(SAMPLE_DIR / 'employees.csv', 'w', encoding='utf-8', newline='') as f:
  writer = csv.DictWriter(f, fieldnames=['이름', '부서', '직급'])
  writer.writeheader()
  writer.writerows(csv_data)

In [ ]:
# RAG: 검색 증강 생성.. Retrieval-Augmented Generation.. - 내부 검색엔진
# AI, 데이터가 없는 상태에서 Hallucination(환각 작용) 생성형 모델들의 착각
# Token 단위로 이해를 함 -> 생성을 할때 토큰 단위로 생성된다? -> 여러 단어중에서 가장 후보가 높은 단어를 취득
# 이름이 ____
#       학생
#       김철수
#       자동차
#       못생겼다 ...
# 이전 문자열이 다음 단어를 시퀀셜하게 생성함. 그래서 더 잘 생성하도록 처리(Embeddings - 벡터화)
# chunking된 데이터, vectorstore에 저장된 데이터(cosine-similarity)를 바탕으로 오차 범위를 줄여줌...

# Fine-tuning: AI를 새로 학습시키는 과정.. - 새로 교육시킴

# RAG의 장점 (Fine-tuning과 비교했을 때), 좀 더 빨리 결과물을 도출. 경제적이고 구현도가 낮음. <<
# 단점은? cosine-similarity에 의해 몇 결과들이 의도했던 맥락과 다름. & 시간/인과관계 등 RAG의 한계점..

In [ ]:
# 그래서 문서 전처리를 빡세게 해야함.

In [ ]:
knowledge_base = [
    {'id': 1, 'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'source': 'programming_guide.txt'},
    {'id': 2, 'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'source': 'ai_glossary.txt'},
    {'id': 3, 'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'source': 'database_manual.txt'},
]

query = 'RAG 기술이 뭐예요?'

def retrieve_by_similarity(query, documents):
  contents = [doc['content'] for doc in documents]

  query_vec = embeddings.embed_query(query)
  doc_vecs = embeddings.embed_documents(contents)

  scores = cosine_similarity([query_vec], doc_vecs)[0]

  ranked = sorted(zip(documents, scores), key = lambda x: x[1], reverse=True)
  return [(doc, float(score)) for doc, score in ranked]

# 가장 기본적인 RAG 처리 방식..!
retrieve_by_similarity(query, knowledge_base)


[({'id': 2,
   'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.',
   'source': 'ai_glossary.txt'},
  0.543921584053737),
 ({'id': 1,
   'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.',
   'source': 'programming_guide.txt'},
  0.2649787322320477),
 ({'id': 3,
   'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.',
   'source': 'database_manual.txt'},
  0.16782349823731557)]

In [ ]:
# 그 전에 python 문법 보고 가자
# class: variables, methods를 하나로 묶음 (청사진)
# class classname:
#   # 변수 메서드 등이 포함됨
#   def __init__():

class Calculator:
  def __init__(self): # ⭐️self⭐️
    self.history = []

  def add(self, a, b):
    result = a + b
    self.history.append(f'{a} + {b} = {result}')
    return result

  def get_history(self):
    return self.history

In [ ]:
calc = Calculator()
calc.add(1,2)

3

In [ ]:
calc.history

['1 + 2 = 3']

In [ ]:
calc.add(3, 4)

7

In [ ]:
calc.history

['1 + 2 = 3', '3 + 4 = 7']

In [ ]:
# 이걸 이용해서 RAG에 사용할 문서를 저장하는 클래스 생성
class DocumentStore:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content': content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

  # 질의와 가장 유사도가 높은 문서를 출력해라 !!
  def retrieve(self, query, count):
    contents = [doc['source'] for doc in self.documents]

    query_vec = embeddings.embed_query(query) # 쿼리 벡터
    doc_vecs = embeddings.embed_documents(contents) # 문서 벡터

    scores = cosine_similarity([query_vec], doc_vecs)[0]

    ranked = sorted(zip(self.documents, scores), key = lambda x: x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked][:count]

In [ ]:
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색 증강 생성이다', 'glossary.txt')
store.add('파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'programming_guide.txt')
store.add('RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'ai_glossary.txt')
store.add('벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'database_manual.txt')

In [ ]:
store.count()

5

In [ ]:
store.search('파이썬')

[{'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'}]

In [ ]:
# 벡터 유사도가 높은 문서를 Return 하는 retrieve 함수를 생성해 보자
query = 'RAG에 대해 설명해 주세요.'
# query = '유사도 검색은 어떻게 하나요?'
store.retrieve(query, 1)

[({'id': 4,
   'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.',
   'source': 'ai_glossary.txt'},
  0.2470238545023874)]

In [ ]:
# 선생님 답안 _ 똑같네.. 허허허
class DocumentStoreT:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content': content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

  # 질의와 가장 유사도가 높은 문서를 출력해라 !! - 선생님 답안
  def retrieve(self, query, top_k = 3):
    if not self.documents:
      return []

    contents = [doc['source'] for doc in self.documents]

    query_vec = embeddings.embed_query(query) # 쿼리 벡터
    doc_vecs = embeddings.embed_documents(contents) # 문서 벡터

    scores = cosine_similarity([query_vec], doc_vecs)[0]

    ranked = sorted(zip(self.documents, scores), key = lambda x: x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked][:top_k]

In [ ]:
# 저장된 모든 문자열들의 단어 수를 카운트하는 클래스
class WordCounter:
  def __init__(self):
    self.store = []

  def add_text(self, text):
    # text 저장
    self.store.append(text)
    return f'{len(text.split())} 단어 저장완료'

  def count_words(self):
    # 현재까지 저장된 모든 문자열들의 단어의 수를 return 합니다.
    # 모든 저장된 텍스트를 하나의 문자열로 합쳐서 글자 수(단어 수로 해석)를 반환합니다.
    # return f'{len("".join(self.store))} 글자 저장완료'
    return sum(len(t.split()) for t in self.store)

# 클래스 인스턴스 생성
counter = WordCounter()

In [ ]:
counter.add_text('저 장 해 볼 까 나')

'6 단어 저장완료'

In [ ]:
counter.count_words()

7

In [ ]:
from collections import Counter

# 저장된 모든 문자열들의 단어 수를 카운트하는 클래스 - 선생님 답안
class WordCounterT:
  def __init__(self):
    self.texts = []

  def add_text(self, text):
    # text 저장
    self.texts.append(text)

  def count_words(self):
    return sum(len(t.split()) for t in self.texts)

  def most_common(self, n):
    all_words = []
    for t in self.texts:
      all_words.extend(t.split())
    return Counter(all_words).most_common(n)

# 클래스 인스턴스 생성
counterT = WordCounterT()

In [ ]:
counterT.add_text('파이썬은 어렵지 않아요~')
counterT.add_text('나는 학교에 갑니다~')
counterT.count_words()
counterT.most_common(3)

[('파이썬은', 2), ('어렵지', 2), ('않아요~', 2)]

In [ ]:
# 가장 중요한 '생성' 부분이 빠졌죠~
# 우리가 가진 knowledge_base를 context로 추가
def rag_with_langchain(query, documents):
  if not documents:
      return '참고할 문서가 없습니다.'
  # documents 매개변수를 사용하여 context를 생성합니다.
  context = '\n'.join(f'- {doc}' for doc in documents)
  messages = [
      SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해 주세요.'),
      HumanMessage(content = f'문서: \n{context}\n\n질문: {query}\n답변:')
  ]
  # 'message'가 아닌 'messages'를 사용합니다.
  response = llm.invoke(messages)
  # LLM이 생성한 문장..!
  return response.content

In [ ]:
docs = [
    'RAG는 검색 증강 생성 기술입니다.',
    '외부 문서를 검색해서 LLM 답변에 활용합니다.'
]
rag_with_langchain('rag가 뭐야?', docs)

'RAG는 검색 증강 생성 기술로, 외부 문서를 검색하여 LLM(대형 언어 모델) 답변에 활용하는 방법입니다.'

In [ ]:
docs = [
    '김치찌개에는 두부를 넣고, 계란을 넣어야 맛있어',
    '마지막에 다 끓이고 나면 5분 정도 찬바람에 식혀 먹어야 맛있어.'
]
rag_with_langchain('김치찌개를 맛있게 끓이는 방법은?', docs)

'김치찌개를 맛있게 끓이는 방법은 다음과 같습니다:\n\n1. 김치찌개에 두부와 계란을 넣어 조리합니다.\n2. 모든 재료를 다 끓인 후, 약 5분 정도 찬바람에 식힌 다음 먹습니다.\n\n이렇게 하면 더욱 맛있는 김치찌개를 즐길 수 있습니다.'

In [ ]:
rag_with_langchain('rag가 뭐야?', [])

'참고할 문서가 없습니다.'

In [ ]:
# generate 포함하여 클래스를 완성하세요
class DocumentStore2:
  def __init__(self):
    self.documents = []
    self.next_id = 1

  def add(self, content, source='unknown'):
    self.documents.append({
        'id': self.next_id,
        'content': content,
        'source': source
    })
    self.next_id += 1

  def count(self):
    return len(self.documents)

  def search(self, keyword):
    return [doc for doc in self.documents if keyword in doc['content']]

  # 질의와 가장 유사도가 높은 문서를 출력해라
  def retrieve(self, query, top_k = 3):
    if not self.documents:
      return []

    contents = [doc['source'] for doc in self.documents]

    query_vec = embeddings.embed_query(query) # 쿼리 벡터
    doc_vecs = embeddings.embed_documents(contents) # 문서 벡터

    scores = cosine_similarity([query_vec], doc_vecs)[0]

    ranked = sorted(zip(self.documents, scores), key = lambda x: x[1], reverse=True)
    return ranked[:top_k]

  def generate(self, query):
    retrieved = self.retrieve(query)
    if not retrieved:
      return '관련 문서가 없습니다.'

    # documents 매개변수를 사용하여 context를 생성합니다.
    context = '\n'.join(f"[문서 {doc[0]['id']}, 출처: {doc[0]['source']} {doc[0]['content']}]" for doc in retrieved)
    messages = [
        SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해 주세요. 출처를 명시해 주세요.'),
        HumanMessage(content = f'문서: \n{context}\n\n질문: {query}\n답변:')
    ]
    # 'message'가 아닌 'messages'를 사용합니다.
    response = llm.invoke(messages)
    # LLM이 생성한 문장..!
    return response.content


In [ ]:
docs_data = [
    {'id': '1', 'content': 'RAG는 검색 증강 생성 기술입니다.'},
    {'id': '2', 'content': '외부 문서를 검색해서 LLM 답변에 활용합니다.'},
    {'id': '3', 'content': '김치찌개에는 두부를 넣고, 계란을 넣어야 맛있어'},
    {'id': '4', 'content': '마지막에 다 끓이고 나면 5분 정도 찬바람에 식혀 먹어야 맛있어.'}
]

store2 = DocumentStore2()

for doc_item in docs_data:
    store2.add(content=doc_item['content'], source=f"doc_{doc_item['id']}")

# query = '김치찌개를 맛있게 끓이는 방법은?'
query = 'RAG에 대해 설명해 주세요?'

# 이제 store2 인스턴스에 문서가 추가되었으므로 generate를 호출할 수 있습니다.
store2.generate(query)

'RAG는 "Retrieval-Augmented Generation"의 약자로, 검색 증강 생성 기술을 의미합니다. 이 기술은 대규모 언어 모델이 데이터를 생성할 때, 외부 데이터 소스를 검색하여 해당 정보를 활용하는 방식입니다. 이를 통해 더 정확하고 관련성 높은 응답을 생성할 수 있으며, 모델의 성능을 향상시키는 데 도움을 줍니다. (출처: 문서 1, doc_1)'

In [ ]:
# 이제 샘플 데이터 말고 문서를 로딩해서 문서를 바탕으로 LLM 답변을 생성한다. (RAG)
doc = Document(
    page_content = '문서 내용...',
    metadata = {'source' : 'file.txt', 'type' : 'policy'}
)

In [ ]:
file_path = SAMPLE_DIR / 'company_policy.txt'

with open(file_path, 'r', encoding='utf-8') as f:
  raw_text = f.read()

In [ ]:
file_path.name

'company_policy.txt'

In [ ]:
raw_text

'주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'

In [ ]:
doc = Document(page_content = raw_text,
               metadata = {
                   'source': file_path.name,
                   'type': 'txt',
                   'char_count': len(raw_text),
                   'line_count': len(raw_text.splitlines())
               })

In [ ]:
doc

Document(metadata={'source': 'company_policy.txt', 'type': 'txt', 'char_count': 356, 'line_count': 19}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n')

In [ ]:
# dir 에 있는 txt 파일을 읽어서 document에 리스트를 추가하는 역할
def load_text_files(directory):
  documents = []
  for fp in sorted(directory.glob('*.txt')):
    text = fp.read_text(encoding='utf=8')
    doc = Document(page_content = text,
                   metadata = {
                       'source': fp.name,
                       'char_count': len(text)
                   })
    documents.append(doc)
  return documents

In [ ]:
all_docs = load_text_files(SAMPLE_DIR)

In [ ]:
# 문서 로드
all_docs

[Document(metadata={'source': 'ai_report.txt', 'char_count': 396}, page_content='2024년 인공지능 산업 동향 보고서\n\n개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.\n\n주요 트렌드\nRAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.\n멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.\nAI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.\n소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.\n\n시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.\n'),
 Document(metadata={'source': 'company_policy.txt', 'char_count': 356}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'),
 Document(metadata={'source': 'product_

In [ ]:
# csv 읽기
def load_csv_file(csv_file_path):
  documents = []
  with open(csv_file_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f) # Dict -> Dictionary
    for i, row in enumerate(reader):
      content = ' | '.join(f'{k}: {v}' for k, v in row.items())
      doc = Document(page_content = content,
                   metadata = {
                       'source': csv_file_path.name,
                       'char_count': len(content)
                   })
      documents.append(doc)
  return documents

In [ ]:
csv_docs = load_csv_file(SAMPLE_DIR / 'employees.csv')
csv_docs

[Document(metadata={'source': 'employees.csv', 'char_count': 27}, page_content='이름: 김철수 | 부서: 개발1팀 | 직급: 대리'),
 Document(metadata={'source': 'employees.csv', 'char_count': 27}, page_content='이름: 김민아 | 부서: 개발1팀 | 직급: 대리'),
 Document(metadata={'source': 'employees.csv', 'char_count': 27}, page_content='이름: 박지민 | 부서: 개발1팀 | 직급: 대리')]

In [ ]:
# json
test_json = {'name' : 'RAG 프로젝트', 'version' : '1.0', 'feature' : ['검색', '생성']}
with open('sample_data/test.json', 'w', encoding='utf-8') as f:
  json.dump(test_json, f, ensure_ascii=False, indent=2)


In [ ]:
with open('sample_data/test.json', 'r', encoding='utf-8') as f:
  data = json.load(f)

content = json.dumps(data, ensure_ascii=False, indent=2)

In [ ]:
data

{'name': 'RAG 프로젝트', 'version': '1.0', 'feature': ['검색', '생성']}

In [ ]:
content

'{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "feature": [\n    "검색",\n    "생성"\n  ]\n}'

In [ ]:
doc = Document(page_content = content, metadata = {'source': 'test.json', 'keys': list(data.keys())})
doc

Document(metadata={'source': 'test.json', 'keys': ['name', 'version', 'feature']}, page_content='{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "feature": [\n    "검색",\n    "생성"\n  ]\n}')

In [ ]:
def load_json_as_document(file_path):
  path = Path(file_path)
  with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

  content = json.dumps(data, ensure_ascii=False, indent=2)
  keys = list(data.keys())
  return Document(page_content = content, metadata = {'source': path.name, 'keys': keys})

In [2]:
# 2026-03-20 수업!
from pathlib import Path
import os
from langchain_core.documents import Document # Import Document class

# Recreate sample_data directory and files if they don't exist
data_dir = Path("sample_data")
data_dir.mkdir(exist_ok=True) # Ensure directory exists

# Define sample texts (copied from xOICExHNx4bz)
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}

# Write sample texts to files (copied from yXrJyhKeyK8O)
for filename, content in sample_texts.items():
  (data_dir / filename).write_text(content, encoding='utf-8')


files = {
  data_dir / "company_policy.txt" : '사내규정',
  data_dir / 'product_manual.txt' : '제품매뉴얼',
  data_dir / 'ai_report.txt' : 'Al보고서'
}
documents = []
for fpath, category in files.items():
  text = fpath.read_text(encoding='utf-8') # Specify encoding for read_text
  for section in text.strip().split ('\n\n'):
    if section.strip():
      documents.append(Document (
        page_content = section.strip(),
        metadata = {'category' : category, 'source' : fpath.name}
      ))


In [6]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

NameError: name 'embeddings' is not defined

In [ ]:
retriever = vectorstore.as_retrever(search_kwargs={'k': 3})